In [19]:
import numpy as np
import pandas as pd
from math import factorial

df = pd.read_csv("full_wc_match_data.csv")
df = df.dropna()

print(df.isnull().sum())

match_id      0
year          0
stage         0
home_team     0
away_team     0
home_score    0
away_score    0
home_rank     0
away_rank     0
result        0
home_gdp      0
home_pop      0
away_gdp      0
away_pop      0
dtype: int64


In [20]:
def normalize(col):
    std = col.std()
    if std == 0:
        return col * 0  # or just return col
    return (col - col.mean()) / std

df["home_gdp"] = normalize(np.log(df["home_gdp"] + 1))
df["away_gdp"] = normalize(np.log(df["away_gdp"] + 1))

df["home_pop"] = normalize(np.log(df["home_pop"] + 1))
df["away_pop"] = normalize(np.log(df["away_pop"] + 1))

df["home_rank"] = normalize(-np.log(df["home_rank"]))
df["away_rank"] = normalize(-np.log(df["away_rank"]))

def transform_features(row):
    rH = row["home_rank"]
    rA = row["away_rank"]

    gH = row["home_gdp"]
    gA = row["away_gdp"]

    pH = row["home_pop"]
    pA = row["away_pop"]

    xH = np.array([1, rH, gH, pH, rA, gA, pA, 1])
    xA = np.array([1, rA, gA, pA, rH, gH, pH, 0])

    return xH, xA


def train_poisson(df, epochs=500, lr=0.0001):
    beta = np.zeros(8)

    for epoch in range(epochs):
        grad = np.zeros_like(beta)

        for _, row in df.iterrows():
            xH, xA = transform_features(row)

            lambdaH = np.exp(np.dot(beta, xH))
            lambdaA = np.exp(np.dot(beta, xA))

            lambdaH = np.clip(lambdaH, 1e-5, 10)
            lambdaA = np.clip(lambdaA, 1e-5, 10)

            kH = row["home_score"]
            kA = row["away_score"]

            grad += (kH - lambdaH) * xH
            grad += (kA - lambdaA) * xA

        beta += lr * grad

    return beta

def poisson_prob(k, lam):
    return (lam**k * np.exp(-lam)) / factorial(k)

def match_outcome_probs(lambdaH, lambdaA, max_goals=10):
    P_home, P_draw, P_away = 0, 0, 0

    for i in range(max_goals+1):
        for j in range(max_goals+1):
            p = poisson_prob(i, lambdaH) * poisson_prob(j, lambdaA)

            if i > j:
                P_home += p
            elif i == j:
                P_draw += p
            else:
                P_away += p

    return P_home, P_draw, P_away

def predict_lambdas(beta, row):
    xH, xA = transform_features(row)

    lambdaH = np.exp(np.dot(beta, xH))
    lambdaA = np.exp(np.dot(beta, xA))

    return lambdaH, lambdaA

def get_actual_result(row):
    if row["home_score"] > row["away_score"]:
        return 0  # home win
    elif row["home_score"] == row["away_score"]:
        return 1  # draw
    else:
        return 2  # away win


def compute_accuracy(df, beta):
    correct = 0

    for _, row in df.iterrows():
        lambdaH, lambdaA = predict_lambdas(beta, row)

        P_home, P_draw, P_away = match_outcome_probs(lambdaH, lambdaA)

        pred = np.argmax([P_home, P_draw, P_away])
        actual = get_actual_result(row)

        if pred == actual:
            correct += 1

    return correct / len(df)




In [21]:
# Train model
beta = train_poisson(df)

beta

array([ 0.02867329,  0.09585283, -0.05195304,  0.07413362, -0.06886508,
       -0.25958236,  0.1308165 ,  0.52609536])

In [22]:
# Evaluate
accuracy = compute_accuracy(df, beta)

print("Accuracy:", accuracy)

Accuracy: 0.5625
